# Chi-Square + Mixed Flow Integration (CIC17 → CIC18)

This notebook evaluates the impact of integrating **mixed network flows** from the target dataset (**GenIDS-CIC17**) into the source dataset (**GenIDS-CIC18**) before training a Machine Learning-based Intrusion Detection System (IDS).

The mixed integration strategy includes:

- **Benign flows**
- **Malicious (D)DoS flows**

The experiment applies **Chi-Square feature selection** before training the model.

## Experimental strategy

1. Load GenIDS-CIC17 and GenIDS-CIC18.
2. Remove non-feature columns and harmonize the datasets.
3. Encode categorical attributes using a shared mapping.
4. Select benign and malicious flows from CIC17 for integration.
5. Remove the selected CIC17 flows from the CIC17 generalization test set to avoid data leakage.
6. Remove the corresponding benign and malicious flows from CIC18 to keep the training set size controlled.
7. Integrate the selected CIC17 flows into CIC18.
8. Apply Min-Max scaling.
9. Apply Chi-Square feature selection.
10. Train an XGBoost classifier.
11. Evaluate the model under:
    - **Intraset scenario**: CIC18 train/test split.
    - **Interset scenario**: CIC17 generalization test.


## 1. Imports and Global Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from typing import Dict, List, Tuple

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, label_binarize
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
    average_precision_score,
    auc,
)

from xgboost import XGBClassifier


In [ ]:
# -------------------------------------------------------------------------
# Experiment configuration
# -------------------------------------------------------------------------

RANDOM_STATE = 42
INTEGRATION_RATE = 0.20  # Experiment 9 configuration reported in the paper.
N_SELECTED_FEATURES = 25

DATA_DIR = Path("../datasets")
SOURCE_DATASET_PATH = DATA_DIR / "GenIDS-CIC18.csv"
TARGET_DATASET_PATH = DATA_DIR / "GenIDS-CIC17.csv"

LABEL_COLUMN = "multiclass"
BENIGN_CLASS_VALUE = 0
DDOS_CLASS_VALUE = 2

print(f"Integration rate: {INTEGRATION_RATE:.0%}")
print(f"Number of selected features: {N_SELECTED_FEATURES}")
print(f"Benign class value: {BENIGN_CLASS_VALUE}")
print(f"(D)DoS class value: {DDOS_CLASS_VALUE}")


## 2. Helper Functions

In [ ]:
def load_dataset(file_path: Path) -> pd.DataFrame:
    """Load a dataset from a CSV file."""
    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {file_path}. Update DATA_DIR or the CSV file name in the configuration cell."
        )
    return pd.read_csv(file_path)


def show_class_distribution(df: pd.DataFrame, label_col: str = "multiclass", title: str = "Dataset") -> None:
    """Print absolute and relative multiclass distributions."""
    print(f"\n{title} - absolute distribution:")
    print(df[label_col].value_counts().sort_index())

    print(f"\n{title} - relative distribution:")
    print(df[label_col].value_counts(normalize=True).sort_index().map("{:.2%}".format))


def drop_non_feature_columns(df: pd.DataFrame, columns_to_drop: List[str]) -> pd.DataFrame:
    """Remove columns that should not be used as predictive features."""
    existing_columns = [col for col in columns_to_drop if col in df.columns]
    return df.drop(columns=existing_columns)


def encode_categorical_columns(
    source_df: pd.DataFrame,
    target_df: pd.DataFrame,
    categorical_columns: List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Encode categorical columns with a shared mapping across source and target datasets."""
    source_df = source_df.copy()
    target_df = target_df.copy()

    for column in categorical_columns:
        if column in source_df.columns and column in target_df.columns:
            encoder = LabelEncoder()
            combined_values = pd.concat(
                [source_df[column].astype(str), target_df[column].astype(str)],
                axis=0,
                ignore_index=True,
            )
            encoder.fit(combined_values)
            source_df[column] = encoder.transform(source_df[column].astype(str))
            target_df[column] = encoder.transform(target_df[column].astype(str))

    return source_df, target_df


def standardize_numeric_dtypes(df: pd.DataFrame, label_col: str = "multiclass") -> pd.DataFrame:
    """Convert numeric feature columns to float64 and the multiclass label to int64."""
    df = df.copy()

    for column in df.columns:
        if column == label_col:
            df[column] = df[column].astype(np.int64)
        elif pd.api.types.is_numeric_dtype(df[column]):
            df[column] = df[column].astype(np.float64)

    return df


def select_class_subset(
    df: pd.DataFrame,
    label_value: int,
    fraction: float,
    label_col: str = "multiclass",
) -> pd.DataFrame:
    """Select the first fraction of flows belonging to one multiclass category."""
    class_flows = df[df[label_col] == label_value]
    n_selected = int(len(class_flows) * fraction)
    return class_flows.iloc[:n_selected].copy()


def build_mixed_flow_subset(
    df: pd.DataFrame,
    fraction: float,
    label_col: str = "multiclass",
) -> pd.DataFrame:
    """Select benign and (D)DoS flows using the same integration rate."""
    benign_subset = select_class_subset(df, BENIGN_CLASS_VALUE, fraction, label_col)
    ddos_subset = select_class_subset(df, DDOS_CLASS_VALUE, fraction, label_col)
    return pd.concat([benign_subset, ddos_subset], axis=0).copy()


def integrate_flows_by_timestamp(
    source_df: pd.DataFrame,
    target_subset: pd.DataFrame,
    source_name: str,
    target_name: str,
    timestamp_col: str = "bidirectional_first_seen_ms",
) -> pd.DataFrame:
    """Integrate selected target flows into the source dataset and sort by timestamp."""
    source_df = source_df.copy()
    target_subset = target_subset.copy()

    source_df["source_dataset"] = source_name
    target_subset["source_dataset"] = target_name

    integrated_df = pd.concat([source_df, target_subset], ignore_index=True)

    if timestamp_col in integrated_df.columns:
        integrated_df = integrated_df.sort_values(by=timestamp_col)

    return integrated_df.drop(columns=["source_dataset"])


def compute_false_alarm_rate(confusion: np.ndarray, class_labels: List[int]) -> Dict[int, float]:
    """Compute one-vs-rest false alarm rate for each class."""
    far_per_class = {}
    for index, class_label in enumerate(class_labels):
        tp = confusion[index, index]
        fp = confusion[:, index].sum() - tp
        fn = confusion[index, :].sum() - tp
        tn = confusion.sum() - (tp + fp + fn)
        far_per_class[class_label] = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    return far_per_class


def evaluate_multiclass_classifier(model, X, y, dataset_name: str, class_labels: List[int]) -> dict:
    """Evaluate a multiclass classifier with macro, per-class, FAR, ROC-AUC, and PR-AUC metrics."""
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)
    cm = confusion_matrix(y, y_pred, labels=class_labels)
    far_per_class = compute_false_alarm_rate(cm, class_labels)

    metrics = {
        "dataset": dataset_name,
        "accuracy": accuracy_score(y, y_pred),
        "precision_macro": precision_score(y, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y, y_pred, average="macro", zero_division=0),
    }

    per_class_precision = precision_score(y, y_pred, labels=class_labels, average=None, zero_division=0)
    per_class_recall = recall_score(y, y_pred, labels=class_labels, average=None, zero_division=0)
    per_class_f1 = f1_score(y, y_pred, labels=class_labels, average=None, zero_division=0)
    for index, class_label in enumerate(class_labels):
        metrics[f"precision_class_{class_label}"] = per_class_precision[index]
        metrics[f"recall_class_{class_label}"] = per_class_recall[index]
        metrics[f"f1_class_{class_label}"] = per_class_f1[index]
        metrics[f"far_class_{class_label}"] = far_per_class[class_label]

    try:
        y_bin = label_binarize(y, classes=class_labels)
        metrics["auc_roc_ovr"] = roc_auc_score(y_bin, y_proba, average="macro", multi_class="ovr")
        metrics["auc_pr_macro"] = average_precision_score(y_bin, y_proba, average="macro")
    except ValueError:
        metrics["auc_roc_ovr"] = np.nan
        metrics["auc_pr_macro"] = np.nan

    print(f"\n=== {dataset_name} Evaluation ===")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y, y_pred, labels=class_labels, digits=4, zero_division=0))

    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels).plot()
    plt.title(f"Confusion Matrix - {dataset_name}")
    plt.tight_layout()
    plt.show()

    y_bin = label_binarize(y, classes=class_labels)
    plt.figure(figsize=(8, 6))
    for index, class_label in enumerate(class_labels):
        if len(np.unique(y_bin[:, index])) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, index], y_proba[:, index])
        plt.plot(fpr, tpr, label=f"Class {class_label} (AUC = {auc(fpr, tpr):.4f})")
    plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"Multiclass ROC Curves - {dataset_name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return metrics


## 3. Load Datasets

In [ ]:
source_df = load_dataset(SOURCE_DATASET_PATH)
target_df = load_dataset(TARGET_DATASET_PATH)

print("Source dataset shape:", source_df.shape)
print("Target dataset shape:", target_df.shape)

In [ ]:
if LABEL_COLUMN not in source_df.columns or LABEL_COLUMN not in target_df.columns:
    raise KeyError(f"Both datasets must contain the '{LABEL_COLUMN}' column.")

show_class_distribution(source_df, label_col=LABEL_COLUMN, title="Source dataset - CIC18")
show_class_distribution(target_df, label_col=LABEL_COLUMN, title="Target dataset - CIC17")


## 4. Preprocessing and Feature Harmonization

In [ ]:
SOURCE_COLUMNS_TO_DROP = [
    "binary", "mapped_label", "date", "hours", "expiration_id",
    "src_ip", "src_mac", "src_oui", "dst_ip", "dst_mac", "dst_oui",
    "ip_version", "vlan_id", "tunnel_id",
]

TARGET_COLUMNS_TO_DROP = [
    "binary", "label", "Timestamp",
    "src_ip", "src_mac", "src_oui", "dst_ip", "dst_mac", "dst_oui",
    "ip_version", "vlan_id", "tunnel_id",
]

source_df = drop_non_feature_columns(source_df, SOURCE_COLUMNS_TO_DROP)
target_df = drop_non_feature_columns(target_df, TARGET_COLUMNS_TO_DROP)

print("Source dataset shape after column removal:", source_df.shape)
print("Target dataset shape after column removal:", target_df.shape)


In [ ]:
CATEGORICAL_COLUMNS = ["application_name", "application_category_name", LABEL_COLUMN]

source_df, target_df = encode_categorical_columns(
    source_df=source_df,
    target_df=target_df,
    categorical_columns=CATEGORICAL_COLUMNS,
)

source_df = standardize_numeric_dtypes(source_df, label_col=LABEL_COLUMN)
target_df = standardize_numeric_dtypes(target_df, label_col=LABEL_COLUMN)

print(source_df.dtypes.value_counts())
print(target_df.dtypes.value_counts())


In [ ]:
# Ensure that both datasets have the same columns and column order.
common_columns = source_df.columns.intersection(target_df.columns)

source_df = source_df[common_columns].copy()
target_df = target_df[common_columns].copy()

print("Number of common columns:", len(common_columns))
print("Source dataset shape:", source_df.shape)
print("Target dataset shape:", target_df.shape)

## 5. Mixed Flow Selection and Data Leakage Prevention

In [ ]:
# Select 20% of benign and (D)DoS target-domain flows, as reported for Experiment 9.
target_mixed_subset = build_mixed_flow_subset(
    df=target_df,
    fraction=INTEGRATION_RATE,
    label_col=LABEL_COLUMN,
)

print("Selected mixed flows from target dataset:", target_mixed_subset.shape)
print(target_mixed_subset[LABEL_COLUMN].value_counts().sort_index())


In [ ]:
# Remove integrated target flows from the target test set to avoid data leakage.
target_test_df = target_df.drop(index=target_mixed_subset.index).copy()

print("Target test dataset after removing integrated flows:", target_test_df.shape)
show_class_distribution(target_test_df, label_col=LABEL_COLUMN, title="Target test dataset - CIC17")


In [ ]:
# Remove corresponding benign and (D)DoS flows from the source dataset to keep its size controlled.
source_mixed_subset = build_mixed_flow_subset(
    df=source_df,
    fraction=INTEGRATION_RATE,
    label_col=LABEL_COLUMN,
)

source_base_df = source_df.drop(index=source_mixed_subset.index).copy()

print("Removed mixed flows from source dataset:", source_mixed_subset.shape)
print(source_mixed_subset[LABEL_COLUMN].value_counts().sort_index())
print("Source dataset after removal:", source_base_df.shape)
show_class_distribution(source_base_df, label_col=LABEL_COLUMN, title="Source base dataset - CIC18")


## 6. Flow Integration

In [ ]:
integrated_source_df = integrate_flows_by_timestamp(
    source_df=source_base_df,
    target_subset=target_mixed_subset,
    source_name="cic18",
    target_name=f"cic17_{int(INTEGRATION_RATE * 100)}_mixed",
    timestamp_col="bidirectional_first_seen_ms",
)

print("Integrated source dataset shape:", integrated_source_df.shape)
show_class_distribution(
    integrated_source_df,
    label_col=LABEL_COLUMN,
    title="Integrated source dataset - CIC18 + CIC17 mixed flows",
)


## 7. Train/Test Split

In [ ]:
X_source = integrated_source_df.drop(columns=[LABEL_COLUMN])
y_source = integrated_source_df[LABEL_COLUMN]

X_target = target_test_df.drop(columns=[LABEL_COLUMN])
y_target = target_test_df[LABEL_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X_source,
    y_source,
    test_size=0.8,
    random_state=RANDOM_STATE,
    stratify=y_source,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("X_target:", X_target.shape)


## 8. Min-Max Scaling

In [ ]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_target_scaled = scaler.transform(X_target)

print("Scaled training set:", X_train_scaled.shape)
print("Scaled intraset test set:", X_test_scaled.shape)
print("Scaled interset test set:", X_target_scaled.shape)

## 9. Chi-Square Feature Selection

In [ ]:
chi2_selector = SelectKBest(score_func=chi2, k=N_SELECTED_FEATURES)

X_train_selected = chi2_selector.fit_transform(X_train_scaled, y_train)
X_test_selected = chi2_selector.transform(X_test_scaled)
X_target_selected = chi2_selector.transform(X_target_scaled)

selected_indices = chi2_selector.get_support(indices=True)
chi2_scores = chi2_selector.scores_

selected_feature_names = [X_train.columns[i] for i in selected_indices]
selected_scores = [chi2_scores[i] for i in selected_indices]

selected_features_df = (
    pd.DataFrame({
        "feature": selected_feature_names,
        "chi2_score": selected_scores,
    })
    .sort_values(by="chi2_score", ascending=False)
    .reset_index(drop=True)
)

print("Selected features:")
display(selected_features_df)

print("X_train_selected:", X_train_selected.shape)
print("X_test_selected:", X_test_selected.shape)
print("X_target_selected:", X_target_selected.shape)

In [ ]:
plt.figure(figsize=(10, 6))
top_features = selected_features_df.head(10).sort_values(by="chi2_score", ascending=True)

plt.barh(top_features["feature"], top_features["chi2_score"])
plt.xlabel("Chi-Square Score")
plt.ylabel("Feature")
plt.title("Top 10 Selected Features by Chi-Square Score")
plt.tight_layout()
plt.show()

## 10. Model Training

In [ ]:
class_labels = sorted(pd.concat([y_train, y_test, y_target]).unique().tolist())
num_classes = len(class_labels)

model = XGBClassifier(
    eval_metric="mlogloss",
    n_estimators=300,
    max_depth=10,
    objective="multi:softprob",
    num_class=num_classes,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.5,
    random_state=RANDOM_STATE,
)

model.fit(X_train_selected, y_train)
print(f"Class labels: {class_labels}")


## 11. Intraset Evaluation: CIC18

In [ ]:
intraset_metrics = evaluate_multiclass_classifier(
    model=model,
    X=X_test_selected,
    y=y_test,
    dataset_name="Intraset - CIC18",
    class_labels=class_labels,
)


## 12. Interset Evaluation: CIC17

In [ ]:
interset_metrics = evaluate_multiclass_classifier(
    model=model,
    X=X_target_selected,
    y=y_target,
    dataset_name="Interset - CIC17",
    class_labels=class_labels,
)


## 13. Summary of Results

In [ ]:
results_df = pd.DataFrame([intraset_metrics, interset_metrics])
display(results_df)

## Notes

- The Chi-Square selector is fitted only on the training subset to avoid data leakage.
- The scaler is fitted only on the training subset and then applied to both intraset and interset test sets.
- Integrated CIC17 mixed flows are removed from the CIC17 test set before evaluation.
- The same proportion of benign and malicious flows is removed from the CIC18 source dataset to control the final training set size.
- This notebook can be reused for 20%, 40%, 60%, and 80% integration scenarios by changing `INTEGRATION_RATE`.
